In [ ]:
import glob
import mne 
import pandas as pd
import numpy as np
import seaborn as sns
import pingouin as pg
import matplotlib.pyplot as plt

#### ANOVA & t-test

In [ ]:
ch = 'Fz'
method = 'peak'

In [ ]:
def create_df(subjs, events_id, ch, time_win, peak_func):

    data_ch = []

    for subj in subjs:
        for event_name in events_id:
            evk = mne.read_evokeds(subj, condition=event_name, verbose=False)

            evk.pick(ch)

            data = evk.data[0][time_win[0]:time_win[1]]

            if peak_func == 'AUC':
                data[data > 0] = 0
                data_ch.append(np.sum(np.abs(data)))
            elif peak_func == 'peak':
                # data[data > 0] = 0
                data_ch.append(np.amin(data))

    subjs_for_df = sorted([int(name.split('-')[0][-3:]) for name in subjs])


    df = pd.DataFrame({
            'Subject': np.repeat(subjs_for_df, len(events_id)),
            'Condition': np.tile(['Looming', 'Flat'], len(subjs_for_df)),
            'uV': data_ch
        })

    return df

In [ ]:
subjs_all = glob.glob('./analysis_800ms_epochs/looming*-ave.fif')
del subjs_all[2]
del subjs_all[5]
print(subjs_all)

subjs = []
for i in subjs_all:
    subjs.append(i.lower())

subjs = sorted(subjs, key=lambda x: int(x.split('-')[0][-3:]))

df = create_df  (
                    subjs_all, 
                    ['1002', '1004'], 
                    f'{ch}', 
                    [500, 800], 
                    f'{method}'
                )

In [ ]:
ax = sns.lineplot(x='Condition', y = 'uV', units= "Subject", estimator=None, data = df, hue="Subject", legend=False)
sns.stripplot(x="Condition", y="uV", hue="Condition", data=df, jitter=False)
ax = sns.pointplot(x='Condition', y = 'uV', data = df)
ax.set_title(f'{ch} {method} between Conditions (500-800ms)')
sns.despine()

In [ ]:
# Repeated measure ANOVA
pg.rm_anova(dv='uV', 
               within='Condition', 
               subject="Subject",
               data=df,
               detailed=True)

In [ ]:
pg.ttest(x=df[df.Condition=='Looming'].uV, y=df[df.Condition=='Receding'].uV, paired=True)

In [ ]:
pg.ttest(x=df[df.Condition=='Looming'].uV, y=df[df.Condition=='Flat'].uV, paired=True)

In [ ]:
pg.ttest(x=df[df.Condition=='Receding'].uV, y=df[df.Condition=='Flat'].uV, paired=True)

#### Peak Latency & Amplitude

In [ ]:
evokeds_files = sorted(glob.glob('./analysis_800ms_epochs/looming*-ave.fif'))
# del evokeds_files[2]
print(evokeds_files)
evks = []

for subj in evokeds_files:
    evks.append(mne.read_evokeds(subj, condition=None, verbose=False))

standard_combined = mne.combine_evoked([e[0] for e in evks], weights='nave')
looming_combined = mne.combine_evoked([e[1] for e in evks], weights='nave')
receding_combined = mne.combine_evoked([e[2] for e in evks], weights='nave')
deviant_combined = mne.combine_evoked([e[3] for e in evks], weights='nave')

all_conds = [standard_combined, looming_combined, receding_combined, deviant_combined]

In [ ]:
# Replace all channels with proper channel names instead of numbers
channel_names_old = looming_combined.ch_names
channel_names_new = ['Fp1','Fz','F3','F7','FT9','FC5','FC1','C3','T7','TP9','CP5','CP1','Pz','P3','P7','O1','Oz','O2','P4','P8','TP10','CP6',
                        'CP2','C4','T8','FT10','FC6','FC2','F4','F8','Fp2', 'AF7','AF3','AFz','F1','F5','FT7','FC3','C1','C5','TP7','CP3','P1','P5',
                        'PO7','PO3','POz','PO4','PO8','P6','P2','CPz','CP4','TP8','C6','C2','FC4','FT8','F6','AF8','AF4','F2','FCz', 'Cz']
channel_dict = dict(zip(channel_names_old, channel_names_new))

for cond in all_conds:
    mne.rename_channels(cond.info, mapping=channel_dict)

In [ ]:
# Function to print out the channel (ch) containing the peak latency and amplitude within the provided time range 
def print_peak_measures(ch, tmin, tmax, lat, amp):
    print(f'Channel: {ch}')
    print(f'Time Window: {tmin * 1e3:.3f} - {tmax * 1e3:.3f} ms')
    print(f'Peak Latency: {lat * 1e3:.3f} ms')
    print(f'Peak Amplitude: {amp * 1e6:.3f} µV')

In [ ]:
# looming_Fz = looming_combined.copy().pick('Fz')
# receding_Fz = receding_combined.copy().pick('Fz')
# deviant_Fz = deviant_combined.copy().pick('Fz')
# tmin, tmax = 0.1, 0.3

# # Get the peak and latency measure from the selected channel
# ch_looming, lat_looming, amp_looming = looming_Fz.get_peak(tmin=tmin, tmax=tmax, mode='neg', return_amplitude=True)
# looming_data = looming_Fz.data[0][int(tmin*1000) : int(tmax*1000)]
# looming_auc = (np.sum(np.abs(looming_data)))
# print('---------------------- Looming ----------------------')
# print_peak_measures(ch_looming, tmin, tmax, lat_looming, amp_looming)
# print(f'AUC: {looming_auc}')

# ch_receding, lat_receding, amp_receding = receding_Fz.get_peak(tmin=tmin, tmax=tmax, mode='neg', return_amplitude=True)
# receding_data = receding_Fz.data[0][int(tmin*1000) : int(tmax*1000)]
# receding_auc = (np.sum(np.abs(receding_data)))
# print('---------------------- Receding ----------------------')
# print_peak_measures(ch_receding, tmin, tmax, lat_receding, amp_receding)
# print(f'AUC: {receding_auc}')

# ch_deviant, lat_deviant, amp_deviant = deviant_Fz.get_peak(tmin=tmin, tmax=tmax, mode='neg', return_amplitude=True)
# deviant_data = deviant_Fz.data[0][int(tmin*1000) : int(tmax*1000)]
# deviant_auc = (np.sum(np.abs(deviant_data)))
# print('---------------------- Deviant ----------------------')
# print_peak_measures(ch_deviant, tmin, tmax, lat_deviant, amp_deviant)
# print(f'AUC: {deviant_auc}')

In [ ]:
diff_loom = mne.combine_evoked([looming_combined, standard_combined],  weights=[1, -1])
diff_rec = mne.combine_evoked([receding_combined, standard_combined],  weights=[1, -1])
diff_dev = mne.combine_evoked([deviant_combined, standard_combined],  weights=[1, -1])

mne.viz.plot_compare_evokeds([diff_loom, diff_rec, diff_dev], picks='Fz', ci=0.95, legend=True, truncate_xaxis=False, invert_y=True, show_sensors=False)

# # Specify times to plot at, as [min],[max],[stepsize]
# times_loom = np.arange(0, diff_loom.tmax, 0.1)
# times_rec = np.arange(0, diff_rec.tmax, 0.1)
# times_dev = np.arange(0, diff_dev.tmax, 0.1)
# diff_loom.plot_topomap(times=times_loom, average=0.050);
# diff_rec.plot_topomap(times=times_rec, average=0.050);
# diff_dev.plot_topomap(times=times_dev, average=0.050);

In [ ]:
# # and order with spectral reordering
# # If you don't have scikit-learn installed set order_func to None
# from sklearn.manifold import spectral_embedding  # noqa
# from sklearn.metrics.pairwise import rbf_kernel   # noqa


# epochs = mne.read_epochs('./analysis_s_markers/Looming004-epo.fif')
# print(len(epochs))

# def order_func(times, data):
#     this_data = data[:, (times > 0.0) & (times < 0.350)]
#     this_data /= np.sqrt(np.sum(this_data ** 2, axis=1))[:, np.newaxis]
#     return np.argsort(spectral_embedding(rbf_kernel(this_data, gamma=1.),
#                       n_components=1, random_state=0).ravel())


# good_pick = 'Fz'  # channel with a clear evoked response
# bad_pick = 'Pz'  # channel with no evoked response

# # We'll also plot a sample time onset for each trial
# plt_times = np.linspace(.15, .25, len(epochs))

# plt.close('all')
# mne.viz.plot_epochs_image(epochs, [good_pick, bad_pick], sigma=.5,
#                           order=order_func, 
#                           show=True)